In [1]:
%load_ext autoreload
%autoreload 1
import typing
from pathlib import Path
import pandas as pd
import numpy as np
from scipy import stats

# DATA_DIR = Path("../../data/v3")
DATA_VERSION = "v2"
INTERMEDIATES_DIR = Path(f"outputs/{DATA_VERSION}/intermediate")

# COMMON COLUMN NAMES
C_STN = "stn_id"
C_FUA = "fua_id"
C_O = "src"
C_D = "trg"
C_FUA_O = f"fua_{C_O}"
C_FUA_D = f"fua_{C_D}"
C_GEO = "geometry"
C_GEOCENTRE = "centre"
C_ACCESS_DIST = "dist_access"
C_ACCESS_TIME = "time_access"
# aliases
C_OD_DIST = "dist_od_stns"
C_OD_SPEED = "speed_od_stns"
C_NTRANS = "n_trans"
C_TTRANS = "trans_time"
C_TIME_IV = "inveh_time"

In [2]:
it_shortest = pd.read_parquet(INTERMEDIATES_DIR /
                              "shortest-itineraries.parquet")
it_2 = pd.read_parquet(INTERMEDIATES_DIR / "itineraries-enriched.parquet")

In [3]:
it_shortest[["trans_time", "trans_time_per"]].describe()

,trans_time,trans_time_per
count,34752.000000,34752.000000
mean,16.822974,10.447586
std,14.791791,9.503024
min,0.000000,0.000000
25%,6.000000,4.000000
50%,13.000000,8.000000
75%,24.000000,15.000000
max,60.000000,60.000000


In [4]:
# it_shortest["tt_per"] = it_shortest[""]

In [5]:
# Build per-country domestic reference statistics with a tight distance band and fallback.
THETA_BAND = 0.125
MIN_DOM_MATCHES = 5


def get_domestic_distbounded_counterfactuals(country,
                                             dist,
                                             subset,
                                             #    domestic_by_country_sorted,
                                             #    metrics: typing.Dict[str, str],
                                             theta=THETA_BAND, min_matches=MIN_DOM_MATCHES,
                                             ) -> typing.Dict:  # TODO update signature
    # gates structurally-infeasible distances and uses an n-closest fallback instead of all-domestic
    _nan_result = {
        "n_dom_matches": 0,
        "used_fallback": np.nan,
        "out_of_range": False,
    }

    # if country not in domestic_by_country_sorted:
    #     return None, _nan_result

    # subset = domestic_by_country_sorted[country]
    dist_values = subset[C_OD_DIST].values

    # structural eligibility gate: country's domestic network can't support this distance
    max_dom_dist = dist_values.max()
    if dist > max_dom_dist * (1 + theta):
        result = dict(_nan_result)
        result["out_of_range"] = True
        result["n_dom_matches"] = np.nan
        # result["used_fallback"] = np.nan
        return None, result

    lo, hi = dist * (1 - theta), dist * (1 + theta)
    idx_lo, idx_hi = np.searchsorted(dist_values, [lo, hi])
    selected = subset.iloc[idx_lo:idx_hi]

    selected_meta = {
        "n_dom_matches": len(selected),
        "used_fallback": np.nan,
        "out_of_range": False,
    }

    return selected, selected_meta


def calculated_popweighted_mean(df: pd.DataFrame, popw_col: str,
                                metric_cols: typing.Sequence[str],
                                ):
    popw_num = df[metric_cols].mul(df[popw_col], axis=0)
    popw_denom = df[popw_col]

    return popw_num.sum()/popw_denom.sum()

# def _pct_rank_speed(row):
#     dom_speeds = domestic_by_country_sorted.get(row["country"])
#     if dom_speeds is None or dom_speeds.empty:
#         return np.nan
#     return stats.percentileofscore(dom_speeds["speed_od_stns"].values, row["intl_speed"])

# new compare

In [6]:

# Prepare domestic and neighbouring per-trip metrics from the shortest itineraries.
# <= 500 # TODO: depreciate, not needed?
dist_threshold = 600
dist_criteria = it_shortest.dist_od_stns <= dist_threshold
domestic_it2 = it_shortest[(it_shortest.adjacency == 0) & dist_criteria].copy()
neighbour_it2 = it_shortest[(
    it_shortest.adjacency == 1) & dist_criteria].copy()
neighbour_it2[["icc1", "icc2"]] = np.sort(
    neighbour_it2[["icc_src", "icc_trg"]], axis=1)

for df in (domestic_it2, neighbour_it2):
    df["speed_od_stns"] = df[C_OD_DIST] / df[C_TIME_IV] * 60
    df["detour"] = df["dist"] / df[C_OD_DIST]
    df["speed_inveh"] = df[C_OD_DIST] / df[C_TIME_IV] * 60
    df["sp_ttime"] = np.where(
        df[C_NTRANS] == 0, np.nan, df[C_TTRANS] / df[C_NTRANS])

domestic_by_country_sorted = {
    c: sub.sort_values(C_OD_DIST)
    for c, sub in domestic_it2.groupby("icc_src", sort=False)
}

In [7]:
domestic_it2.describe()

,path_id,src,trg,dep_win,time,gcd,dist,inveh_time,trans_time,n_trans,...,popu_src,popu_trg,popu_od_sum,popu_od_sqsum,popu_od_logsum,popu_od_prod,popu_od_eucmean,popu_od_geomean,detour,sp_ttime
count,1.259700e+04,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,...,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,12597.000000,8304.000000
mean,3.256793e+06,1852.177820,1918.660872,0.817814,234.764865,327.579344,422.959355,224.838454,9.926411,0.943717,...,0.781839,0.781613,1.563452,1.539016,-0.626862,0.568780,1.292886,0.586886,1.623899,11.549795
std,3.963964e+06,2503.673877,2543.821609,0.781194,130.257396,196.141873,259.567288,124.929673,11.197485,0.788790,...,1.327977,1.328276,1.855587,0.607194,0.487872,1.573876,1.754549,0.473670,1.059813,8.744222
min,0.000000e+00,0.000000,0.000000,0.000000,9.000000,6.000000,50.000000,9.000000,0.000000,0.000000,...,0.200022,0.200022,0.401623,0.896240,-1.394424,0.040325,0.283991,0.200811,0.954725,2.500000
25%,6.685010e+05,134.000000,142.000000,0.000000,137.000000,178.000000,230.000000,131.000000,0.000000,0.000000,...,0.266188,0.266188,0.665775,1.143793,-0.991284,0.102027,0.483577,0.319417,1.254263,5.500000
50%,1.398775e+06,446.000000,500.000000,1.000000,222.000000,302.000000,387.000000,212.000000,7.000000,1.000000,...,0.407730,0.407730,0.922053,1.337146,-0.728073,0.187037,0.682923,0.432477,1.393508,9.000000
75%,4.600417e+06,2866.000000,3015.000000,1.000000,315.000000,447.000000,572.000000,302.000000,14.000000,2.000000,...,0.685476,0.685476,1.628603,1.719483,-0.347455,0.449309,1.339162,0.670305,1.664912,15.000000
max,2.408730e+07,9226.000000,9226.000000,2.000000,1011.000000,1528.000000,2472.000000,991.000000,60.000000,2.000000,...,12.877820,12.877820,15.874691,5.319715,1.586510,38.593165,13.221932,6.212340,56.021890,60.000000


In [8]:
neighbour_it2.columns

Index(['path_id', 'src', 'trg', 'dep_win', 'time', 'gcd', 'dist', 'inveh_time',
       'trans_time', 'n_trans', 'gtt', 'speed', 'fua_src', 'fua_trg',
       'dist_od_stns', 'speed_od_stns', 'icc_src', 'icc_trg', 'adjacency',
       'circuity_osm_od', 'circuity_osm_leg', 'speed_inveh', 'speed_net',
       'trans_time_per', 'popu_src', 'popu_trg', 'popu_od_sum',
       'popu_od_sqsum', 'popu_od_logsum', 'popu_od_prod', 'popu_od_eucmean',
       'popu_od_geomean', 'icc1', 'icc2', 'detour', 'sp_ttime'],
      dtype='object')

# gather relative comparisons between countries

In [9]:
# need country by symmetric pair, not OD
# - add the sorted columns by neighbour_od, encode, then sort by? or take the merged form?

# %%
# produces 2 rows per intl trip via O-role / D-role melt
metrics = dict(
    speed_ivod="speed_inveh", speed_ivot="speed", speed_net="speed_net",
    dist_od="dist_od_stns", dist_track="dist", detour="circuity_osm_od",
    ntrans="n_trans", ttrans="trans_time",
    spttrans="trans_time_per",
)
popu_cols = ["popu_od_sum", "popu_od_prod"]

# check all input columns are present
assert not (missing_cols := [c for c in metrics.values() if c not in domestic_it2.columns]
            ), f"Columns {missing_cols} expected but not present in domestic df set, expected any within {domestic_it2.columns}"
metric_cols = list(metrics.values())

map_idx_other = {"icc_src": "icc_trg", "icc_trg": "icc_src"}

multiple mean from population weighting

verification: get the number of matches (or no matches) per country pair

need to infer the "other" country


# full domestic compare

In [10]:
# ============================================================================
# PHASE 1: Build domestic reference metric arrays (pre-compute per country)
# ===========================================================================
%aimport lib.relative_compare
import lib.relative_compare as rel

dom_metric_arrays, dom_pop_weights_by_country = rel.build_domestic_metric_arrays(
    domestic_by_country_sorted, metric_cols
)

print(
    f"✓ Phase 1: Built domestic reference arrays for {len(dom_metric_arrays)} countries")
for c in list(dom_metric_arrays.keys())[:3]:
    print(f"  {c}: {len(dom_metric_arrays[c][metric_cols[0]])} domestic ODs")

✓ Phase 1: Built domestic reference arrays for 24 countries
  DE: 6300 domestic ODs
  DK: 12 domestic ODs
  UK: 2409 domestic ODs


In [11]:
# PHASE 2: Compute per-IOD dual deltas (vs all domestic ODs)

iod_deltas_dict = rel.compute_iod_deltas_dual_reference(
    neighbour_it2,
    domestic_by_country_sorted,
    dom_metric_arrays,
    dom_pop_weights_by_country,
    metric_cols
)

print(f"✓ Phase 2: Computed deltas for {len(iod_deltas_dict)} country pairs")
sample_pair = list(iod_deltas_dict.keys())[0]
print(
    f"  Sample pair {sample_pair}: {len(iod_deltas_dict[sample_pair])} reference countries")

# PHASE 2b: Single-stage raw-metric pair means (mirrored rows, no domestic reference)
df_intl_means = rel.compute_intl_pair_means(neighbour_it2, metric_cols)

print(f"✓ Phase 2b: Computed raw intl pair means for {df_intl_means['pair'].nunique()} pairs "
      f"({len(df_intl_means)} rows, mirrored)")
print(df_intl_means[["pair", "reference_country", "other_country",
                     "n_iods", "pop_coverage", metric_cols[0]]].head(4))

✓ Phase 2: Computed deltas for 36 country pairs
  Sample pair ('AT', 'CH'): 2 reference countries
✓ Phase 2b: Computed raw intl pair means for 36 pairs (72 rows, mirrored)
    pair reference_country other_country  n_iods  pop_coverage  speed_inveh
0  AT-CH                AT            CH      56     31.353901    63.348256
1  AT-CH                CH            AT      56     31.353901    63.348256
2  AT-CZ                AT            CZ      50     45.851771    50.644777
3  AT-CZ                CZ            AT      50     45.851771    50.644777


In [12]:
# PHASE 3: Aggregate per-IOD deltas to pair level
df_pair_agg, df_iod_level = rel.aggregate_iod_deltas_to_pairs(
    iod_deltas_dict, metric_cols)

print(
    f"✓ Phase 3: Aggregated to {len(df_pair_agg)} pair-reference combinations")
print(f"  IOD-level rows: {len(df_iod_level)}")
print(f"  Metrics: {metric_cols}")

✓ Phase 3: Aggregated to 72 pair-reference combinations
  IOD-level rows: 20382
  Metrics: ['speed_inveh', 'speed', 'speed_net', 'dist_od_stns', 'dist', 'circuity_osm_od', 'n_trans', 'trans_time', 'trans_time_per']


In [13]:
# PHASE 4: Add IOD-level diagnostics

df_pair_agg = rel.add_iod_diagnostics(df_pair_agg, df_iod_level)

print(f"✓ Phase 4: Added IOD-level diagnostics (n_iods, pop_coverage, pop_var)")
print(f"  Sample row:")
print(df_pair_agg[["pair", "reference_country",
      "n_iods", "pop_coverage"]].iloc[0])

✓ Phase 4: Added IOD-level diagnostics (n_iods, pop_coverage, pop_var)
  Sample row:
pair                     AT-CH
reference_country           AT
n_iods                    56.0
pop_coverage         31.353901
Name: 0, dtype: object


In [14]:
# PHASE 5: Secondary diagnostic (domestic reference bias inspection)

df_domestic_diag = rel.compute_domestic_reference_diagnostics(
    domestic_by_country_sorted)

print(f"✓ Phase 5: Generated domestic reference diagnostics (inspection only)")
print(f"  Countries: {len(df_domestic_diag)}")
print(f"  High variance countries (var > 1e12):")
high_var = df_domestic_diag[df_domestic_diag["pop_var"]
                            > 1e12].sort_values("pop_var", ascending=False)
print(high_var.head())

# PHASE 6: Export and summary

# Export pair-level aggregates
output_path = INTERMEDIATES_DIR / "pairwise-deltas-alldom-v1.parquet"
df_pair_agg.to_parquet(output_path, index=False)
print(f"✓ Phase 6: Exported {len(df_pair_agg)} rows to {output_path.name}")

# Export per-country population-weighted domestic means
dom_popweighted_means = rel.compute_domestic_popweighted_means(
    domestic_by_country_sorted, neighbour_it2, metric_cols)
dom_popweighted_means.to_parquet(
    INTERMEDIATES_DIR / "domestic-popweighted-means-v1.parquet")
print(f"✓ Phase 6: Exported {len(dom_popweighted_means)} countries to domestic-popweighted-means-v1.parquet "
      f"({dom_popweighted_means.notna().sum().sum()} non-NaN cells)")


# Print summary statistics
print(f"\nSummary:")
print(f"  Total pairs: {len(df_pair_agg.groupby('pair'))}")
print(f"  Total rows (pair × reference): {len(df_pair_agg)}")
print(f"  Metrics per row: {len(metric_cols)}")
print(
    f"  Output schema: [pair, reference_country, other_country, {', '.join(metric_cols[:2])}..., n_iods, pop_coverage, pop_var]")

# Preview
print(f"\nPreview (first 3 rows):")
print(df_pair_agg[["pair", "reference_country",
      "other_country", "n_iods", metric_cols[0]]].head(3))

# PHASE 6b: Export single-stage intl pair means
output_path_means = INTERMEDIATES_DIR / "intl-popweighted-pair-means-v1.parquet"
df_intl_means.to_parquet(output_path_means, index=False)
print(
    f"✓ Phase 6b: Exported {len(df_intl_means)} rows to {output_path_means.name}")

# Sanity check: mirrored rows must be identical
mirror_check = df_intl_means.groupby("pair")[metric_cols].nunique().max().max()
assert mirror_check == 1, "Mirrored rows differ — symmetry violated"
print("✓ Mirror symmetry verified: both rows per pair carry identical metric values")

✓ Phase 5: Generated domestic reference diagnostics (inspection only)
  Countries: 24
  High variance countries (var > 1e12):
Empty DataFrame
Columns: [pop_coverage, pop_var]
Index: []
✓ Phase 6: Exported 72 rows to pairwise-deltas-alldom-v1.parquet
✓ Phase 6: Exported 24 countries to domestic-popweighted-means-v1.parquet (189 non-NaN cells)

Summary:
  Total pairs: 36
  Total rows (pair × reference): 72
  Metrics per row: 9
  Output schema: [pair, reference_country, other_country, speed_inveh, speed..., n_iods, pop_coverage, pop_var]

Preview (first 3 rows):
    pair reference_country other_country  n_iods  speed_inveh
0  AT-CH                AT            CH    56.0   -16.181969
1  AT-CH                CH            AT    56.0   -11.309408
2  AT-CZ                AT            CZ    50.0   -28.885449
✓ Phase 6b: Exported 72 rows to intl-popweighted-pair-means-v1.parquet
✓ Mirror symmetry verified: both rows per pair carry identical metric values


In [15]:
df_pair_agg.describe()

,speed_inveh,speed,speed_net,dist_od_stns,dist,circuity_osm_od,n_trans,trans_time,trans_time_per,n_iods,pop_coverage,pop_var
count,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,66.000000,65.000000
mean,-22.830519,-18.813882,-21.045026,102.677413,266.125568,0.317419,0.770963,10.727988,6.854466,305.757576,228.122399,5.195665
std,25.611051,39.908707,40.199720,112.917065,279.627185,0.861244,0.493516,10.001662,7.887119,507.107212,351.653612,19.047211
min,-87.468049,-113.366411,-115.343793,-136.513743,-647.219143,-1.621117,-0.870572,-41.276784,-41.276784,1.000000,0.438637,0.000000
25%,-41.720998,-38.908209,-41.981757,45.681382,97.198104,0.015540,0.515922,6.478534,4.320216,30.000000,10.109503,0.239509
50%,-21.891406,-19.677681,-21.350970,92.784962,256.511072,0.193990,0.762780,10.119484,6.690425,97.500000,54.545941,1.170716
75%,-4.940833,7.221849,3.283971,162.405578,432.357811,0.547053,1.027623,14.251160,9.815983,397.000000,295.028811,2.642638
max,52.960042,81.559167,81.655393,387.638994,912.415539,2.796755,2.000000,30.500000,21.138284,2351.000000,1435.821698,110.575233


In [16]:
len(df_pair_agg.reference_country.unique())

24

In [17]:
df_pair_agg["trans_time"] / df_pair_agg["n_trans"]

0     14.267304
1     15.011573
2     19.059612
3     16.917516
4      8.594299
        ...    
67          NaN
68     7.866884
69     6.952928
70    20.524247
71    19.058478
Length: 72, dtype: float64

In [18]:
df_intl_means.describe()

,speed_inveh,speed,speed_net,dist_od_stns,dist,circuity_osm_od,n_trans,trans_time,trans_time_per,n_iods,pop_coverage,pop_var
count,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,72.000000,70.000000
mean,64.861261,110.746422,107.312831,320.591736,604.107839,1.980375,0.967220,13.389613,9.406740,283.083333,210.558687,4.857036
std,20.994554,28.781463,28.290517,100.009945,242.992242,0.780882,0.518552,7.886188,5.233478,491.245177,341.608935,18.386045
min,19.495885,71.506851,71.506849,131.371508,185.000000,1.322068,0.000000,0.000000,0.000000,1.000000,0.438637,0.000000
25%,51.545354,87.002987,85.576800,262.879003,474.180710,1.483104,0.578476,8.532392,6.541966,23.000000,9.982495,0.229130
50%,62.946420,104.519762,100.031331,332.616461,548.375059,1.731827,1.098415,12.430095,8.858636,68.500000,45.166783,0.863542
75%,74.551659,130.504347,126.882087,407.868672,763.861781,2.093146,1.332147,17.525590,13.095792,336.250000,273.005497,2.585985
max,123.381782,174.470790,169.123094,495.849362,1186.777593,4.073687,2.000000,30.500000,21.908968,2351.000000,1435.821698,110.575233


In [19]:
dom_popweighted_means.describe()

,speed_inveh,speed,speed_net,dist_od_stns,dist,circuity_osm_od,n_trans,trans_time,trans_time_per
count,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000,21.000000
mean,84.113454,122.970643,121.725679,214.647407,360.750406,1.707737,0.221933,4.366816,4.130291
std,26.771513,35.942735,35.805482,97.346714,243.413444,0.786350,0.238655,9.782918,9.776969
min,20.029636,79.683220,79.542730,54.408383,69.187130,1.112413,0.000000,0.000000,0.000000
25%,70.421740,96.226418,94.499725,145.255469,232.000000,1.330304,0.042646,0.539808,0.539808
50%,79.530225,117.049179,117.049180,211.185699,315.311068,1.455468,0.189746,1.967659,1.967659
75%,101.536226,142.653061,141.970214,295.208196,423.000000,1.763197,0.312285,3.270413,2.754452
max,138.883952,203.451761,203.052416,380.824814,1149.000000,4.446899,1.000000,46.000000,46.000000
